# Module 11: Instrumental Variables

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

An instrument is a variable that pushes units into treatment and affects the
outcome **only** through that push. When one exists it identifies an effect
without any parallel trends assumption at all.

None exists in these records. This module teaches the conditions, tests every
candidate the dataset contains, and shows that the closest to relevant is also
the most clearly invalid.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

In [ ]:
X = profile.set_index("agency_id").copy()
X = X.loc[sorted(X.index)]
X["treated"] = [1 if a in TRAINED else 0 for a in X.index]
X["lpop"] = np.log(X["population_served"])
X["lsworn"] = np.log(X["sworn_officers"])
print(f"{len(X)} agencies, {X['treated'].sum()} of them treated")

## 2. The three conditions

> **Relevance.** The instrument predicts treatment. Testable, and the usual
> bar is a first stage F statistic above 10.
>
> **Exclusion.** The instrument affects the outcome only through treatment.
> **Not testable**, ever, from the data alone.
>
> **Monotonicity.** Nobody is pushed out of treatment by the instrument.
> Not testable either.

Two of the three cannot be tested. That is the trade instrumental variables
offers: it replaces an assumption you can check weakly with assumptions you
cannot check at all, in exchange for not needing parallel trends.

**A partial test of exclusion is available here**, and it is the one to run:
if a candidate is associated with the outcome **before** the program existed,
it is affecting the outcome through some route other than a treatment that had
not happened yet.

## 3. Every candidate the dataset holds

In [ ]:
X["east"] = (X["region"] == "East").astype(int)
X["sheriff"] = (X["agency_type"] == "County Sheriff").astype(int)
X["lbudget"] = np.log(X["county_budget_millions"])
X["lcounty"] = np.log(X["county_population"])
pre = f[f["period"] == "before"]
X["pre_rate"] = pre.groupby("agency_id").apply(
    lambda g: 100 * g["n_uof"].sum() / g["n_arrests"].sum())

rows = []
for lab, c in [("region is East", "east"), ("is a sheriff's office", "sheriff"),
               ("county budget", "lbudget"), ("county population", "lcounty"),
               ("public safety budget share", "budget_share_public_safety_pct")]:
    z1 = sm.OLS(X["treated"], sm.add_constant(X[[c]].astype(float))).fit()
    z2 = sm.OLS(X["pre_rate"], sm.add_constant(X[[c]].astype(float))).fit()
    rows.append({"candidate": lab,
                 "first stage F": round(z1.fvalue, 2),
                 "relevance p": round(z1.pvalues.iloc[1], 3),
                 "associated with the pre program outcome, p":
                     round(z2.pvalues.iloc[1], 3)})
pd.DataFrame(rows).set_index("candidate")

**Not one candidate reaches an F statistic of 1**, let alone 10. The strongest
is region, at 1.60.

And region is also the candidate most clearly associated with the outcome
**before the program existed**, at p = 0.067. Eastern agencies in this state
are smaller and rural, and their use of force rates differ for reasons that
have nothing to do with a training program.

**The candidate closest to relevant is the one most clearly failing
exclusion.** That is not a coincidence. Variables that predict which agencies
get a program usually do so because they capture something about those
agencies, and that something usually affects the outcome directly.

## 4. What happens if you use one anyway

In [ ]:
import warnings as _w
_w.filterwarnings("ignore")

agg = f[f["period"].isin(["before", "after"])].groupby(
    ["agency_id", "period"]).apply(
    lambda g: 100 * g["n_uof"].sum() / g["n_arrests"].sum()).unstack()
agg["change"] = 100 * (agg["after"] / agg["before"] - 1)
agg["treated"] = [1 if a in TRAINED else 0 for a in agg.index]
agg["east"] = X.loc[agg.index, "east"]

first = sm.OLS(agg["treated"], sm.add_constant(agg[["east"]])).fit()
reduced = sm.OLS(agg["change"], sm.add_constant(agg[["east"]])).fit()
wald = reduced.params["east"] / first.params["east"]
print(f"  first stage, east on treated:      {first.params['east']:+.3f}  "
      f"F = {first.fvalue:.2f}")
print(f"  reduced form, east on the change:  {reduced.params['east']:+.2f} points")
print(f"  the ratio, a Wald instrumental variables estimate: {wald:+.1f} percent")
print(f"\n  the truth: {TRUTH:+.1f} percent")

The ratio comes out at 22.8 percent against a truth of 12, nearly double.

**Notice what it is not: absurd.** A weak instrument estimate that came back
at 400 percent would be caught by anyone. This one looks like a plausible
policy effect, and a reader given only the number has no way to tell that its
denominator, the first stage coefficient, is not distinguishable from zero.

That is the weak instrument pathology, and it does not announce itself in the
estimate. **The first stage F statistic is what announces it**, which is why
it is reported alongside, and why an F of 1.60 ends the analysis rather than
starting it.

## 5. What a usable instrument would look like here

| Property | What it would need to be |
|---|---|
| Relevance | something that pushed agencies into the program, strongly |
| Exclusion | with no other route to use of force |
| Monotonicity | never pushing an agency out |
| Plausible candidates | a lottery among eligible agencies; a funding formula with an arbitrary cutoff; a trainer's travel schedule |

**All three of those are design features, not data features.** They exist when
someone builds them in, which is the argument in Beginner
[Topic 16](../../Beginner/Topic_16_Randomness_Solves_A_Problem.md) for
randomising the first wave: it creates an instrument where none existed.

Searching an administrative dataset for an instrument after the fact almost
never succeeds, and the searching itself is a specification problem: with
enough candidates, one will pass the relevance test by chance.

## Exercise

Confirm that searching for an instrument finds one eventually, even when none
exists.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rng = np.random.default_rng(11)
    best = []
    for _ in range(300):
        z = rng.normal(size=len(X))
        fs = sm.OLS(X["treated"], sm.add_constant(z)).fit()
        best.append(fs.fvalue)
    best = np.array(best)
    print("  300 candidate instruments, every one pure noise\n")
    print(f"    largest first stage F found: {best.max():.2f}")
    print(f"    candidates with F above 4:   {int((best > 4).sum())} "
          f"({100 * np.mean(best > 4):.0f} percent)")
    print(f"    candidates with F above 10:  {int((best > 10).sum())}")
    print(f"\n  for comparison, the best real candidate had F = 1.60")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Among three hundred pure noise candidates, a handful clear an F of 4 and the
largest goes higher still. **None of them is an instrument, and one of them
would have passed the only testable condition.**

That is the danger of instrument hunting. Relevance is the condition you can
test, so it is the one a search optimises, and passing it says nothing about
the two conditions that matter and cannot be tested.

The discipline that protects against this is the same one as everywhere else
in this series: **name the instrument and the argument for exclusion before
looking at the data**, and report how many candidates were considered. An
instrument found by searching needs a much stronger substantive story than one
specified in advance, and usually does not have one.

</details>

---

**Next:** [Module 12: Regression Discontinuity](Module_12_Regression_Discontinuity.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*